## Анализ коммерческих показателей маркетплейса

**Цель исследования:** Оценить общий коммерческий оборот (GMV) маркетплейса, выявить ключевые драйверы выручки среди категорий товаров и провести аудит стоимости логистики для поиска операционных аномалий.

**Используемый стек:** Python, Pandas.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PATH_TO_DATA = '/content/drive/MyDrive/Datasets/olist-ecommerce-analytics/data/'

In [ ]:
import pandas as pd

In [ ]:
df_orders = pd.read_csv(PATH_TO_DATA + 'olist_orders_dataset.csv')
df_items = pd.read_csv(PATH_TO_DATA + 'olist_order_items_dataset.csv')
df_products = pd.read_csv(PATH_TO_DATA + 'olist_products_dataset.csv')

# Подготовка и очистка данных

In [ ]:
print("Заказы загружены. Формат:", df_orders.shape)
print("Позиции загружены. Формат:", df_items.shape)

Заказы загружены. Формат: (99441, 8)
Позиции загружены. Формат: (112650, 7)


In [ ]:
# Распределение статусов
print("*** СТАТУСЫ ЗАКАЗОВ ***")
print('\n')
print(df_orders['order_status'].value_counts())

*** СТАТУСЫ ЗАКАЗОВ ***


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [ ]:
# Проверка пропусков в таблице заказов
print("*** ПРОПУСКИ В ТАБЛИЦЕ ЗАКАЗОВ (df_orders) ***")
print('\n')
print(df_orders.isnull().sum())

*** ПРОПУСКИ В ТАБЛИЦЕ ЗАКАЗОВ (df_orders) ***


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [ ]:
# Проверка уникальности ключей
print("*** ПРОВЕРКА НА ДУБЛИКАТЫ ***")
print('\n')
print(f"Всего строк в df_orders: {len(df_orders)}")
print(f"Уникальных order_id в df_orders: {df_orders['order_id'].nunique()}")

*** ПРОВЕРКА НА ДУБЛИКАТЫ ***


Всего строк в df_orders: 99441
Уникальных order_id в df_orders: 99441


In [ ]:
df_orders_delivered = df_orders[df_orders['order_status'] == 'delivered']

In [ ]:
# Проверка на пропуски уже в отфильрованной таблице со статуром 'delivered'

print("*** Пропуски в отфильтованной таблице (df_orders_delivered) ***")
print('\n')
print(df_orders_delivered.isnull().sum())

*** Пропуски в отфильтованной таблице (df_orders_delivered) ***


order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


**Аналитическая заметка: проверка качества данных**
>
В ходе первичного анализа отфильтрованных данных (только со статусом `delivered`) были обнаружены аномалии: **8 заказов** успешно доставлены по документам, но не имеют физической даты вручения клиенту (`order_delivered_customer_date` равен `NaN`).
>
**Возможные причины:**
> 1. *Человеческий фактор:* Курьер фактически передал посылку, но забыл закрыть заявку в мобильном приложении.
> 2. *Технический сбой:* Ошибка при синхронизации баз данных
>
**Бизнес-решение:** Данные 8 строк были удалены из выборки. Оставление их в датасете может привести при последующем расчете операционной эффективности к ошибкам в коде или исказят средние показатели работы логистики.

In [ ]:
# Оставляею только доставленные И только те, у которых дата доставки НЕ пустая

df_orders_clean = df_orders_delivered.dropna(subset=['order_delivered_customer_date'])

# Подготовка ветрин данных путем объеденения таблиц

In [ ]:
# Объеденяю очищенную таблицу 'df_orders_clean' с заказами с таблицей поизиций товаров 'df_items', а также с таблицей df_products для вывода категорий товаров

df_sales_full = df_orders_clean.merge(
    df_items,
    on='order_id',
    how='inner'
).merge(
    df_products,
    on='product_id',
    how='inner'
)

In [ ]:
df_sales_full.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0


In [ ]:
# Расчет общего оборота бизнеса (GMV)

total_gmv = df_sales_full['price'].sum()
print(f"Общий GMV маркетплейса: {total_gmv:,.2f} реалов")

Общий GMV маркетплейса: 13,220,248.93 реалов


In [ ]:
print('*** ТОП-10 самых прибыльных категорий товаров***')
print('\n')
print(df_sales_full.groupby('product_category_name')['price'].sum().sort_values(ascending=False).head(10))

*** ТОП-10 самых прибыльных категорий товаров***


product_category_name
beleza_saude              1233131.72
relogios_presentes        1165898.98
cama_mesa_banho           1023434.76
esporte_lazer              954673.55
informatica_acessorios     888613.62
moveis_decoracao           711927.69
utilidades_domesticas      615628.69
cool_stuff                 610204.10
automotivo                 578849.35
brinquedos                 471097.49
Name: price, dtype: float64


In [ ]:
# Расчет процента "Золотых доставок"

df_expensive_delivery = df_sales_full[df_sales_full['freight_value'] >= df_sales_full['price']]
print(f"Процент заказов с дорогой доставкой {len(df_expensive_delivery) / len(df_sales_full) * 100:.2f} %")

Процент заказов с дорогой доставкой 3.64 %


## Финальный бизнес-отчет: Экономика продаж и аудит логистики

В ходе анализа коммерческих показателей маркетплейса на основе объединенной витрины данных были получены следующие результаты:

### 1. Ключевые метрики (KPI)
*   **Общий оборот (GMV):** **13.22 млн** бразильских реалов.
*   **Доля «золотой доставки»:** **3.64%** от общего числа заказов. Стоимость логистики (`freight_value`) в этих операциях превышает или равна стоимости самого товара.

### 2. Структура выручки (Топ-3 драйвера)
Основную долю коммерческого оборота генерируют три категории товаров:
1.  **beleza_saude:** 1.23 млн реалов.
2.  **relogios_presentes:** 1.16 млн реалов.
3.  **cama_mesa_banho:** 1.02 млн реалов.

> **Резюме для коммерческого департамента:** Топ-3 категории суммарно приносят **более 25% всей выручки** маркетплейса. Рекомендуется перераспределить marketing budget в пользу этих направлений для максимизации прибыли.

### 3. Риски и операционные аномалии
Показатель "золотой доставки" в 3.64% удерживается в рамках нормы, однако требует операционного контроля.
*   **Гипотеза:** Данный эффект может быть вызван продажей низкомаржинальных дешевых товаров на дальние расстояния либо некорректной работой алгоритма расчета региональных тарифов.
*   **Рекомендация:** Передать список этих транзакций в отдел логистики для пересмотра минимальной стоимости заказа или оптимизации тарифной сетки для удаленных регионов.

#Задача: Географический анализ клиентской базы маркетплейса

##Контекст бизнеса:

Маркетинговый департамент планирует запуск крупных региональных рекламных кампаний. Чтобы не слить бюджет впустую, им нужно четкое понимание: какие регионы Бразилии уже сейчас приносят компании больше всего денег, а какие находятся в аутсайдерах.


Определить топ-5 ключевых штатов Бразилии по объему выручки (GMV) на основе исторических данных о продажах и составить профиль географического распределения клиентов.

In [ ]:
# Загрузка таблицы с информауией о клиентах

df_customers = pd.read_csv(PATH_TO_DATA + 'olist_customers_dataset.csv')

In [ ]:
# Соеденение таблицы с клиентами с нашей ветриной данных df_sales_full

df_sales_full = df_sales_full.merge(
    df_customers,
    on='customer_id',
    how='inner')

In [ ]:
df_sales_full.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,268.0,4.0,500.0,19.0,8.0,13.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,178.0,1.0,400.0,19.0,13.0,19.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,232.0,1.0,420.0,24.0,19.0,21.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO


In [ ]:
df_states = df_sales_full.groupby('customer_state')['price'].sum().reset_index().rename(columns={'price': 'state_GMV'})
df_states['share %'] = round(df_states['state_GMV'] / total_gmv * 100, 2)

top_5_states_by_gmv = df_states.sort_values(by='state_GMV', ascending=False).reset_index(drop=True).head(5)

print('*** Топ-5 штатов по обороту с долей от общего оборота маркетплейса ***')
print('\n')
print(top_5_states_by_gmv)

*** Топ-5 штатов по обороту с долей от общего оборота маркетплейса ***


  customer_state   state_GMV  share %
0             SP  5066562.98    38.32
1             RJ  1759651.13    13.31
2             MG  1552481.83    11.74
3             RS   728718.47     5.51
4             PR   666063.51     5.04


Определить топ-5 ключевых штатов Бразилии по среднему чеку (AOV)

In [ ]:
sum_revenue = df_sales_full.groupby(['order_id', 'customer_state'])['price'].sum().reset_index()
states_aov = sum_revenue.groupby('customer_state')['price'].mean().reset_index().rename(columns={'price': 'state_AOV'})
top_5_states_by_aov = states_aov.sort_values(by='state_AOV', ascending=False).reset_index(drop=True).head(5).round(2)

print('*** Топ-5 штатов по среднему чек (AOV) ***')
print('\n')
print(top_5_states_by_aov)

*** Топ-5 штатов по среднему чек (AOV) ***


  customer_state  state_AOV
0             PB     217.77
1             AP     199.62
2             AC     199.14
3             AL     198.63
4             RO     187.99


# Бизнес инсайт

Если посмтреть на две таблицы одновременно, то становится видно географическое распределение бизнеса на 2 зоны где:


*   **SP, RJ, MG** генерируют почти 64 % выручки от всей компании. НО средний чек низкий и даже не попадает в Топ-5 компаний по среднему чеку. В этой зоне бизнес растет за счет плотности населения и более развитой логистики, там клиенты заказывают чаще и легко более недорогие товары.
*   **PB, AP, AC** это зона редкого и дорого спроса. Общая доля невелика, но каждый заказ приносит в среднем на 20%-40% болбше денег чем чек из Сан-Паулу (SP).

** Главный вывод:
Маркетплейс зависит от своей сверх-лояльной аудитории из первой зоны. Однако потенциал маржинальности и роста прибыльности скрыт в удаленных регионах. Бизнес совершит ошибку, если будет оценивать эффективность маркетинга в отдаленных штатах только по «общей выручке», игнорируя их уникально высокий средний чек.
